# Retrotraducción (R6): traducir paráfrasis al shiwilu con el checkpoint NLLB-200 + LoRA (F. Prado)

`3_baselines_y_aumento_datos/tecnicas_aumento/retrotraduccion.py` necesita este checkpoint para traducir las paráfrasis en español al shiwilu (paso 2 de la técnica).

**Antes de correr:**
1. `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> **GPU**.
2. En tu computadora, haz `git commit` + `git push` de TODO lo último (en especial `2_baselines/comun.py` y `2_baselines/split_fijo.csv`). Esta Colab clona tu repo desde GitHub: si no hiciste push, usaría un split distinto y el aumento quedaría contaminado con oraciones de test.
3. Ten el checkpoint en Drive: `Mi unidad/shiwilu_checkpoint/nllb_bidi_lora_v2_1b_loraplus_xl/`, con `adapter_model.safetensors` (exactamente ese nombre, sin "Copia de"), `adapter_config.json` y los archivos del tokenizer.

**Qué celdas correr (de arriba abajo):**
- Pasos **1, 2, 3, 4A, 5 y 6**.
- El paso **4B (entrenar desde cero)** es OPCIONAL: sáltalo si tienes el checkpoint en Drive. Solo se usa si lo perdiste (tarda 30 min - 2 h).

**IMPORTANTE:** si Colab reinicia el entorno o algo falla con `ModuleNotFoundError` / `No such file or directory`, vuelve a correr desde el paso 1, no solo la celda que falló.

**Cuándo hay que volver a correr este notebook:** cada vez que cambie el split train/dev/test, porque las paráfrasis se generan a partir de las oraciones de train.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonar tu repo y el de F. Prado (requiere haber hecho `git push`)

In [ ]:
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git clone https://github.com/fapi19/Tesis_Spa-Jeb.git 3_baselines_y_aumento_datos/tecnicas_aumento/tesis_spa_jeb
!git log --oneline -3

import os
assert os.path.exists('/content/shiwilu-tesis/2_baselines/split_fijo.csv'), (
    "Falta 2_baselines/split_fijo.csv en GitHub: haz git commit + git push desde tu "
    "computadora y vuelve a correr esta celda."
)
print('OK: split_fijo.csv presente (split congelado).')

## 3. Instalar dependencias en un entorno aislado (las que F. Prado ya fijó y probó)

In [ ]:
%cd /content/shiwilu-tesis/3_baselines_y_aumento_datos/tecnicas_aumento/tesis_spa_jeb
# Creamos un entorno virtual AISLADO del Python de Colab. Colab trae mas de
# 100 paquetes preinstalados (jax, tensorflow, opencv, wandb, etc.) con sus
# propias versiones de numpy/protobuf, y al instalar los requisitos de F.
# Prado encima de eso se generaban conflictos que dejaban TODO el paquete
# `transformers` roto por dentro. En un entorno aislado, ninguno de esos
# paquetes preinstalados esta presente, asi que no hay con que chocar.
#
# Usamos `virtualenv` (no el modulo estandar `venv`) porque `venv` falla en
# Colab al intentar instalar pip dentro del entorno nuevo (paso `ensurepip`
# roto en la distribucion de Ubuntu que usa Colab). `virtualenv` trae su
# propio pip empaquetado y no depende de ese paso.
!pip install -q virtualenv
!rm -rf /content/nmt_venv
!virtualenv -q /content/nmt_venv
!/content/nmt_venv/bin/pip install -q -r requirements/nmt.txt
!/content/nmt_venv/bin/pip install -q sentence-transformers
!/content/nmt_venv/bin/python -c "import torch; print('GPU disponible:', torch.cuda.is_available())"
print('Entorno listo en /content/nmt_venv. De aqui en adelante, las celdas usan /content/nmt_venv/bin/python en vez de python normal.')

## 4A. Copiar el checkpoint desde Drive al disco local (recomendado)

Se copia al disco de la sesión porque leer el modelo directo desde Drive falla a veces (`HFValidationError: Repo id must be in the form...`). La celda verifica que estén los archivos clave antes de seguir.

In [ ]:
import os

CHECKPOINT = "/content/checkpoint_retro/nllb_bidi_lora_v2_1b_loraplus_xl"

!mkdir -p /content/checkpoint_retro
!cp -r "/content/drive/MyDrive/shiwilu_checkpoint/nllb_bidi_lora_v2_1b_loraplus_xl" /content/checkpoint_retro/
for nombre in sorted(os.listdir(CHECKPOINT)):
    ruta = os.path.join(CHECKPOINT, nombre)
    print(f"{os.path.getsize(ruta)/1e6:10.1f} MB  {nombre}" if os.path.isfile(ruta) else f"{'':>13}  {nombre}/")

for f in ["adapter_model.safetensors", "adapter_config.json", "tokenizer_config.json"]:
    ruta = os.path.join(CHECKPOINT, f)
    assert os.path.exists(ruta), (
        f"Falta {f} en {CHECKPOINT}. En Drive, el archivo debe llamarse exactamente "
        f"{f} (sin 'Copia de') y estar en la carpeta principal del checkpoint, no dentro de checkpoint-XXXX."
    )
tam = os.path.getsize(os.path.join(CHECKPOINT, "adapter_model.safetensors")) / 1e9
assert tam > 1.5, f"adapter_model.safetensors pesa solo {tam:.2f} GB (deberia ser ~1.97 GB): la copia quedo incompleta, vuelve a correr esta celda."
print(f"OK: checkpoint completo ({tam:.2f} GB) en {CHECKPOINT}")

## 4B. (OPCIONAL) Entrenar desde cero — SOLO si NO tienes el checkpoint en Drive

**Salta de aquí hasta el paso 5 si hiciste el 4A.** Entrenar la configuración campeona (v2.1b LoRA+) puede tardar 30 min - 2 horas según la GPU que te toque.

In [ ]:
%cd /content/shiwilu-tesis/3_baselines_y_aumento_datos/tecnicas_aumento/tesis_spa_jeb
!/content/nmt_venv/bin/python -m scripts.nmt.30_train_lora \
    --variant xl \
    --rank 32 \
    --alpha 64 \
    --loraplus-lr-ratio 16 \
    --output-dir models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl

CHECKPOINT = "/content/shiwilu-tesis/3_baselines_y_aumento_datos/tecnicas_aumento/tesis_spa_jeb/models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl"

### 4B (cont.) Guardar el checkpoint en Drive (¡no te saltes este paso si entrenaste!)

In [ ]:
%cd /content/shiwilu-tesis/3_baselines_y_aumento_datos/tecnicas_aumento/tesis_spa_jeb
!mkdir -p /content/drive/MyDrive/shiwilu_checkpoint
!cp -r models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl /content/drive/MyDrive/shiwilu_checkpoint/

### 4B (cont.) Evaluar el checkpoint (debería dar chrF++ promedio cercano a 43.2; la vez anterior dio 43.19)

In [ ]:
%cd /content/shiwilu-tesis/3_baselines_y_aumento_datos/tecnicas_aumento/tesis_spa_jeb
%env MPLBACKEND=Agg
!/content/nmt_venv/bin/python -m scripts.nmt.40_evaluate --variant xl --checkpoint models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl --split test

## 5. Correr la retrotraducción

Usa la variable `CHECKPOINT` definida en el paso 4A (o 4B), que pasa a Python directamente (no dentro de un comando `!`). Hace parafraseo en español (Helsinki-NLP), traducción al shiwilu (NLLB+LoRA) y los filtros de calidad.

**El paso 2/3 (traducción) no imprime progreso**: puede tardar 10-30 minutos sin mostrar nada nuevo. No significa que esté colgado; puedes ver el uso de GPU en el gráfico de recursos de Colab.

In [ ]:
import os, subprocess

assert "CHECKPOINT" in globals(), "Falta correr el paso 4A (define la variable CHECKPOINT)."
print("Usando checkpoint:", CHECKPOINT)

proc = subprocess.Popen(
    ["/content/nmt_venv/bin/python",
     "3_baselines_y_aumento_datos/tecnicas_aumento/retrotraduccion.py",
     "--checkpoint", CHECKPOINT],
    cwd="/content/shiwilu-tesis",
    env={**os.environ, "MPLBACKEND": "Agg"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for linea in proc.stdout:
    print(linea, end="")
proc.wait()
assert proc.returncode == 0, "retrotraduccion.py fallo: revisa el error de arriba."

## 6. Descargar el resultado

Descarga solo el CSV (no el checkpoint, pesa varios GB). Luego, en tu computadora, reemplaza `3_baselines_y_aumento_datos/tecnicas_aumento/salidas/retrotraduccion.csv` con el archivo descargado y avísale a Claude para re-correr las evaluaciones.

In [ ]:
%cd /content/shiwilu-tesis
import pandas as pd
from google.colab import files

ruta = '3_baselines_y_aumento_datos/tecnicas_aumento/salidas/retrotraduccion.csv'
df = pd.read_csv(ruta)
print(len(df), 'filas')
print(df['estado_filtro'].value_counts())
files.download(ruta)